# Four thousand draws, four hundred of them real

A sampler finished. There are four chains and a thousand draws each, the trace plot looks
like a caterpillar, and the posterior mean has four decimal places. Two of those decimals are
knowledge and the other two are where the chains happened to be standing.

Split-R̂, bulk and tail ESS, and MCSE following Vehtari et al. (2021), in numpy. `diagnose`
produces a `ConvergenceReport` with the thresholds it used; non-finite draws fail the verdict
outright. `to_inference_data` hands a posterior to ArviZ when it is installed.

The verdict is **per parameter**, and one bad row is enough — a fit is not "mostly converged"
any more than a bridge is mostly attached.

In [ ]:
import numpy as np

from axiom.core import is_failure
from axiom.infer import (
    ConvergenceReport, ConvergenceThresholds, ParameterDiagnostics, Posterior, diagnose, ess_bulk, ess_tail,
    from_inference_data, mcse_mean, split_rhat, to_inference_data,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import CRITICAL, caption, compare, lines
enable();  # every axiom result renders itself from here on

In [ ]:
rng = np.random.default_rng(0)
good = rng.normal(size=(4, 1000))
bad = good + np.array([0.0, 0.0, 1.5, 1.5])[:, None]
ar = np.zeros((4, 1000))
for c in range(4):
    for t in range(1, 1000):
        ar[c, t] = 0.9 * ar[c, t - 1] + rng.normal() * np.sqrt(1 - 0.81)
table(
    [
        [name, f"{split_rhat(d):.3f}", f"{ess_bulk(d):.1f}", f"{ess_tail(d):.1f}", f"{mcse_mean(d):.4f}"]
        for name, d in (("iid", good), ("shifted", bad), ("AR(0.9)", ar))
    ],
    headers=("draws", "split R-hat", "ESS bulk", "ESS tail", "MCSE"),
)

### Two failures that look nothing alike

`shifted` is two chains exploring one place and two exploring another: the marginal looks
bimodal-ish, every chain is individually smooth, and the mean is an average of two answers.
`AR(0.9)` is one honest chain moving too slowly: it will get there, but there are not four
thousand independent draws in it and any error bar computed as if there were is four times
too narrow.

In [ ]:
window = slice(0, 200)
fig = lines(
    np.arange(200),
    {f"chain {c}": bad[c, window] for c in range(4)},
    title="Split R-hat catches this",
    subtitle="four chains of the same parameter — each one smooth, two of them somewhere else",
    x_title="draw", y_title="value",
)
caption(fig, "Nothing within a chain is wrong. The disagreement is between chains, which is "
             "exactly the quantity R-hat compares — and why running one long chain cannot "
             "detect it.")

In [ ]:
fig = lines(
    np.arange(200),
    {"iid": good[0, window], "AR(0.9)": ar[0, window]},
    title="…and effective sample size catches this",
    subtitle="one chain each, same length, same marginal variance",
    x_title="draw", y_title="value",
)
caption(fig, "The slow chain visits the same places; it just takes many draws to get "
             "anywhere new. ESS is the count of draws that would have been worth as much.")

In [ ]:
counts = {"iid": ess_bulk(good), "shifted": ess_bulk(bad), "AR(0.9)": ess_bulk(ar)}
fig = compare(
    [f"{k}  ({v:,.0f} of 4,000)" for k, v in counts.items()], list(counts.values()),
    highlight=f"AR(0.9)  ({counts['AR(0.9)']:,.0f} of 4,000)",
    value_fmt="{:,.0f}",
    title="What four thousand draws were worth",
    subtitle="bulk effective sample size out of 4 chains × 1,000 draws",
    x_title="effective draws",
)
caption(fig, "Every Monte-Carlo standard error in the package divides by the bottom number "
             "rather than by 4,000. Reporting to four decimals from the last row is reporting "
             "the sampler's path, not the posterior.")

In [ ]:
post = Posterior({"mu": good, "tau": np.exp(ar)}, provenance={"seed": 0, "divergences": 0})
report: ConvergenceReport = diagnose(post)
print(report.converged, report.thresholds)
print(report.to_frame())
row: ParameterDiagnostics = report.row("tau")
print(row.passes(ConvergenceThresholds(rhat_max=1.05, ess_min=100.0)))
loose = diagnose(post, ess_min=100.0)
print("with ess_min=100:", loose.converged, loose.failing)

The report carries **the thresholds it used**. A verdict of "converged" that does not say
against what is not a verdict, and `ess_min` is exactly the knob somebody lowers at 6pm on a
Friday — so it is recorded next to the answer.

In [ ]:
idata = to_inference_data(post)
if not is_failure(idata):
    back = from_inference_data(idata)
    print(type(idata).__name__, back.names(), back == post)
else:
    print(idata)

## Seeing it

`enable()` at the top of this notebook already made a bare result on the last
line of a cell render itself — a card drawn by `rich`, or the same content as
aligned plain text where `rich` is not installed. `show` does it on demand, for
a result that is not the last thing in its cell.

`axiom.viz` draws the figure this subpackage's results are actually about.

In [ ]:
import numpy as np

from axiom.core import Posterior
from axiom.display import show
from axiom.infer import diagnose
from axiom.viz import convergence

rng = np.random.default_rng(0)
posterior = Posterior({"alpha": rng.normal(size=(4, 600)), "beta": rng.normal(size=(4, 600))})
report = diagnose(posterior)
show(report)
convergence(report)

The question a sampler's output poses is *may I use this*, and the answer is per parameter:
one bad row is enough.

## What this bought you

Two failure modes that a trace plot flatters — chains that disagree, and chains that crawl —
turned into numbers with stated thresholds, per parameter, in the four dependencies of the
core install.